# Triton: Python-like GPU Programming

## Historical Context

**Triton** was introduced by OpenAI in 2021 as a language and compiler for writing custom GPU kernels in Python.

### Timeline
- **2021**: Triton 1.0 released by OpenAI (Philippe Tillet)
- **2022**: Triton 2.0 with improved compiler optimizations
- **2022**: PyTorch integrates Triton via `torch.compile`
- **2023**: Flash Attention 2 uses Triton extensively
- **2024**: Triton becomes standard for custom kernels in PyTorch

### Why Triton?

**Before Triton:**
- CUDA: Powerful but complex, requires deep hardware knowledge
- CuPy/Numba: Limited optimization capabilities
- Gap between research ideas and efficient implementation

**Triton's Innovation:**
- **Python-like syntax**: Familiar to ML researchers
- **Automatic optimization**: Compiler handles memory coalescing, shared memory
- **Block-level programming**: Think in tiles, not threads
- **Portable**: Works across NVIDIA, AMD (experimental)

### Key Advantages over CUDA
1. **10x less code** for equivalent kernels
2. **Automatic memory management** (coalescing, bank conflicts)
3. **Easier to optimize** (tune block sizes, not thread synchronization)
4. **Faster iteration** (no separate compilation step)
5. **Comparable performance** (often within 5-10% of hand-tuned CUDA)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Optional

# Check if Triton is available
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
    print(f"Triton version: {triton.__version__}")
except ImportError:
    TRITON_AVAILABLE = False
    print("Triton not available. Install with: pip install triton")
    print("Note: Triton requires CUDA-capable GPU")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Triton Programming Model

### Key Concepts

1. **Programs operate on blocks**: Think in terms of tiles of data
2. **Automatic parallelization**: Triton compiler distributes blocks across GPU
3. **Explicit memory loads/stores**: Control what data is loaded into fast memory
4. **Pointer arithmetic**: Similar to C, but with safety checks

### Triton vs CUDA

| Aspect | CUDA | Triton |
|--------|------|--------|
| Unit of work | Thread | Block (tile) |
| Memory management | Manual (shared mem) | Automatic |
| Synchronization | Explicit (syncthreads) | Automatic |
| Code complexity | High | Low |
| Performance control | Fine-grained | Coarse-grained |
| Learning curve | Steep | Gentle |

## Your First Triton Kernel: Vector Addition

In [ ]:
if TRITON_AVAILABLE:
    @triton.jit
    def add_kernel(
        x_ptr,  # Pointer to first input vector
        y_ptr,  # Pointer to second input vector
        output_ptr,  # Pointer to output vector
        n_elements,  # Number of elements
        BLOCK_SIZE: tl.constexpr,  # Compile-time constant
    ):
        """
        Triton kernel for vector addition: output = x + y
        
        Each program (block) handles BLOCK_SIZE elements.
        Triton automatically:
        - Coalesces memory accesses
        - Vectorizes operations
        - Manages shared memory
        """
        # Get program ID (which block are we?)
        pid = tl.program_id(axis=0)
        
        # Calculate block start offset
        block_start = pid * BLOCK_SIZE
        
        # Create array of offsets within this block
        offsets = block_start + tl.arange(0, BLOCK_SIZE)
        
        # Create mask for boundary checking
        mask = offsets < n_elements
        
        # Load data (only valid elements)
        x = tl.load(x_ptr + offsets, mask=mask)
        y = tl.load(y_ptr + offsets, mask=mask)
        
        # Compute
        output = x + y
        
        # Store result
        tl.store(output_ptr + offsets, output, mask=mask)
    
    def add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """Wrapper function to launch Triton kernel."""
        assert x.shape == y.shape
        assert x.is_cuda and y.is_cuda
        
        n_elements = x.numel()
        output = torch.empty_like(x)
        
        # Choose block size (power of 2, typically 256-1024)
        BLOCK_SIZE = 1024
        
        # Calculate grid size (number of blocks)
        grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
        
        # Launch kernel
        add_kernel[grid](
            x, y, output, n_elements,
            BLOCK_SIZE=BLOCK_SIZE
        )
        
        return output
    
    # Test the kernel
    x = torch.randn(1000000, device='cuda')
    y = torch.randn(1000000, device='cuda')
    
    output_triton = add(x, y)
    output_torch = x + y
    
    print(f"Max difference: {(output_triton - output_torch).abs().max().item():.2e}")
    print("Triton kernel works correctly!")

## More Complex Example: Fused Softmax

Softmax requires:
1. Find max (for numerical stability)
2. Compute exp(x - max)
3. Sum exponentials
4. Divide by sum

**Standard PyTorch**: 4 separate kernel launches, 8 memory passes
**Triton**: 1 kernel, 2 memory passes (read input, write output)

In [ ]:
if TRITON_AVAILABLE:
    @triton.jit
    def softmax_kernel(
        output_ptr,
        input_ptr,
        input_row_stride,
        output_row_stride,
        n_cols,
        BLOCK_SIZE: tl.constexpr
    ):
        """
        Fused softmax kernel.
        
        Each program handles one row of the input matrix.
        Performs all softmax operations in a single pass:
        1. Load row
        2. Find max
        3. Compute numerically stable softmax
        4. Store result
        """
        # Each program handles one row
        row_idx = tl.program_id(0)
        
        # Calculate row start pointer
        row_start_ptr = input_ptr + row_idx * input_row_stride
        
        # Column offsets
        col_offsets = tl.arange(0, BLOCK_SIZE)
        input_ptrs = row_start_ptr + col_offsets
        
        # Mask for boundary checking
        mask = col_offsets < n_cols
        
        # Load row
        row = tl.load(input_ptrs, mask=mask, other=-float('inf'))
        
        # Find max for numerical stability
        row_max = tl.max(row, axis=0)
        
        # Subtract max and exponentiate
        row_minus_max = row - row_max
        numerator = tl.exp(row_minus_max)
        
        # Sum exponentials
        denominator = tl.sum(numerator, axis=0)
        
        # Compute softmax
        softmax_output = numerator / denominator
        
        # Write result
        output_row_start_ptr = output_ptr + row_idx * output_row_stride
        output_ptrs = output_row_start_ptr + col_offsets
        tl.store(output_ptrs, softmax_output, mask=mask)
    
    def softmax(x: torch.Tensor) -> torch.Tensor:
        """Triton softmax implementation."""
        n_rows, n_cols = x.shape
        
        # Block size must be power of 2 and >= n_cols
        BLOCK_SIZE = triton.next_power_of_2(n_cols)
        
        # Limit to avoid excessive resource usage
        if BLOCK_SIZE > 4096:
            BLOCK_SIZE = 4096
        
        # Allocate output
        y = torch.empty_like(x)
        
        # Launch kernel (one program per row)
        softmax_kernel[(n_rows,)](
            y,
            x,
            x.stride(0),
            y.stride(0),
            n_cols,
            BLOCK_SIZE=BLOCK_SIZE
        )
        
        return y
    
    # Test softmax
    x = torch.randn(1000, 512, device='cuda')
    
    y_triton = softmax(x)
    y_torch = torch.softmax(x, dim=-1)
    
    print(f"Max difference: {(y_triton - y_torch).abs().max().item():.2e}")
    print(f"Mean difference: {(y_triton - y_torch).abs().mean().item():.2e}")
    
    # Verify properties
    print(f"\nRow sums (should be ~1.0):")
    print(f"  Triton: {y_triton.sum(dim=1)[:5]}")
    print(f"  PyTorch: {y_torch.sum(dim=1)[:5]}")

## Benchmark: Triton vs PyTorch Softmax

In [ ]:
if TRITON_AVAILABLE:
    def benchmark_softmax(n_rows, n_cols, n_iter=100):
        """Benchmark softmax implementations."""
        x = torch.randn(n_rows, n_cols, device='cuda')
        
        # Warmup
        for _ in range(10):
            _ = softmax(x)
            _ = torch.softmax(x, dim=-1)
        torch.cuda.synchronize()
        
        # Benchmark Triton
        start = time.time()
        for _ in range(n_iter):
            _ = softmax(x)
        torch.cuda.synchronize()
        time_triton = (time.time() - start) / n_iter * 1000
        
        # Benchmark PyTorch
        start = time.time()
        for _ in range(n_iter):
            _ = torch.softmax(x, dim=-1)
        torch.cuda.synchronize()
        time_torch = (time.time() - start) / n_iter * 1000
        
        return time_triton, time_torch
    
    # Benchmark different sizes
    configs = [
        (1000, 128),
        (1000, 512),
        (1000, 2048),
        (4096, 512),
        (8192, 1024),
    ]
    
    results = []
    print("Softmax Benchmark (Triton vs PyTorch)\n")
    print(f"{'Shape':<20} {'Triton (ms)':<15} {'PyTorch (ms)':<15} {'Speedup':<10}")
    print("-" * 60)
    
    for n_rows, n_cols in configs:
        time_triton, time_torch = benchmark_softmax(n_rows, n_cols)
        speedup = time_torch / time_triton
        results.append((n_rows, n_cols, time_triton, time_torch, speedup))
        
        print(f"{f'{n_rows}x{n_cols}':<20} {time_triton:<15.3f} {time_torch:<15.3f} {speedup:<10.2f}x")

In [ ]:
if TRITON_AVAILABLE and results:
    # Visualize results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    labels = [f"{r}x{c}" for r, c, _, _, _ in results]
    triton_times = [t for _, _, t, _, _ in results]
    torch_times = [p for _, _, _, p, _ in results]
    speedups = [s for _, _, _, _, s in results]
    
    # Time comparison
    x = np.arange(len(labels))
    width = 0.35
    ax1.bar(x - width/2, triton_times, width, label='Triton', alpha=0.8)
    ax1.bar(x + width/2, torch_times, width, label='PyTorch', alpha=0.8)
    ax1.set_xlabel('Matrix Shape')
    ax1.set_ylabel('Time (ms)')
    ax1.set_title('Softmax Performance Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=45)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Speedup
    ax2.bar(x, speedups, alpha=0.8)
    ax2.axhline(y=1, color='r', linestyle='--', alpha=0.5, label='Baseline')
    ax2.set_xlabel('Matrix Shape')
    ax2.set_ylabel('Speedup (Triton / PyTorch)')
    ax2.set_title('Triton Speedup over PyTorch')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels, rotation=45)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('triton_softmax_benchmark.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nKey Insights:")
    print("1. Triton typically faster due to kernel fusion")
    print("2. Speedup increases with problem size (better amortization)")
    print("3. Single kernel launch vs multiple in PyTorch")
    print("4. Reduced memory bandwidth (2 passes vs 8)")

## Flash Attention in Triton

**Flash Attention** (Dao et al., 2022) is the killer application for Triton.

### Standard Attention Problem
```python
# Standard attention (memory-inefficient)
scores = Q @ K.T  # [B, N, N] - huge memory for long sequences
probs = softmax(scores / sqrt(d))
output = probs @ V
```

**Memory**: O(N^2) for attention matrix
**For N=2048**: ~16GB just for attention scores!

### Flash Attention Innovation
- **Tiled computation**: Never materialize full attention matrix
- **Recomputation**: Trade compute for memory
- **IO-aware**: Minimize HBM reads/writes

**Memory**: O(N) - only store output and stats
**Speedup**: 2-4x faster + 10x less memory

In [ ]:
if TRITON_AVAILABLE:
    @triton.jit
    def flash_attention_forward(
        Q, K, V, O,
        stride_qz, stride_qh, stride_qm, stride_qk,
        stride_kz, stride_kh, stride_kn, stride_kk,
        stride_vz, stride_vh, stride_vn, stride_vk,
        stride_oz, stride_oh, stride_om, stride_ok,
        Z, H, N_CTX, P_SEQ,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
        BLOCK_DMODEL: tl.constexpr,
    ):
        """
        Simplified Flash Attention kernel (forward pass only).
        
        Key ideas:
        1. Process attention in blocks (tiles)
        2. Never materialize full attention matrix
        3. Use online softmax algorithm
        4. Minimize HBM accesses
        
        This is a teaching implementation - production Flash Attention
        has many more optimizations.
        """
        # Get program IDs
        start_m = tl.program_id(0)
        off_hz = tl.program_id(1)
        
        # Initialize offsets
        offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = tl.arange(0, BLOCK_N)
        offs_d = tl.arange(0, BLOCK_DMODEL)
        
        # Initialize pointers to Q, K, V
        q_ptrs = Q + off_hz * stride_qh + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qk
        k_ptrs = K + off_hz * stride_kh + offs_n[None, :] * stride_kn + offs_d[:, None] * stride_kk
        v_ptrs = V + off_hz * stride_vh + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vk
        
        # Load Q block (stays in shared memory)
        q = tl.load(q_ptrs, mask=offs_m[:, None] < N_CTX, other=0.0)
        
        # Initialize accumulator and running max/sum for online softmax
        acc = tl.zeros([BLOCK_M, BLOCK_DMODEL], dtype=tl.float32)
        m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float('inf')
        l_i = tl.zeros([BLOCK_M], dtype=tl.float32)
        
        # Loop over K, V blocks
        for start_n in range(0, N_CTX, BLOCK_N):
            # Load K, V blocks
            k = tl.load(k_ptrs + start_n * stride_kn, 
                       mask=offs_n[None, :] + start_n < N_CTX, other=0.0)
            v = tl.load(v_ptrs + start_n * stride_vn,
                       mask=offs_n[:, None] + start_n < N_CTX, other=0.0)
            
            # Compute attention scores for this block
            qk = tl.dot(q, k)  # [BLOCK_M, BLOCK_N]
            qk = qk * (1.0 / tl.sqrt(BLOCK_DMODEL.to(tl.float32)))
            
            # Online softmax: update running max
            m_ij = tl.max(qk, 1)
            m_i_new = tl.maximum(m_i, m_ij)
            alpha = tl.exp(m_i - m_i_new)
            
            # Compute probabilities
            p = tl.exp(qk - m_i_new[:, None])
            
            # Update running sum
            l_i = l_i * alpha + tl.sum(p, 1)
            
            # Update accumulator
            acc = acc * alpha[:, None]
            acc += tl.dot(p.to(v.dtype), v)
            
            m_i = m_i_new
        
        # Final normalization
        acc = acc / l_i[:, None]
        
        # Write output
        o_ptrs = O + off_hz * stride_oh + offs_m[:, None] * stride_om + offs_d[None, :] * stride_ok
        tl.store(o_ptrs, acc, mask=offs_m[:, None] < N_CTX)
    
    def flash_attention(
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
    ) -> torch.Tensor:
        """
        Flash Attention wrapper.
        
        Args:
            q, k, v: [batch, heads, seq_len, head_dim]
        
        Returns:
            output: [batch, heads, seq_len, head_dim]
        """
        batch, heads, seq_len, head_dim = q.shape
        
        # Allocate output
        o = torch.empty_like(q)
        
        # Tile sizes
        BLOCK_M = 64
        BLOCK_N = 64
        
        # Launch kernel
        grid = (triton.cdiv(seq_len, BLOCK_M), batch * heads)
        
        flash_attention_forward[grid](
            q, k, v, o,
            q.stride(0), q.stride(1), q.stride(2), q.stride(3),
            k.stride(0), k.stride(1), k.stride(2), k.stride(3),
            v.stride(0), v.stride(1), v.stride(2), v.stride(3),
            o.stride(0), o.stride(1), o.stride(2), o.stride(3),
            batch, heads, seq_len, seq_len,
            BLOCK_M=BLOCK_M,
            BLOCK_N=BLOCK_N,
            BLOCK_DMODEL=head_dim,
        )
        
        return o
    
    # Test Flash Attention
    batch, heads, seq_len, head_dim = 2, 8, 512, 64
    
    q = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    k = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    v = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    
    # Flash Attention
    o_flash = flash_attention(q, k, v)
    
    # Standard attention
    scale = 1.0 / (head_dim ** 0.5)
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale
    probs = torch.softmax(scores, dim=-1)
    o_standard = torch.matmul(probs, v)
    
    print(f"Max difference: {(o_flash - o_standard).abs().max().item():.2e}")
    print(f"Mean difference: {(o_flash - o_standard).abs().mean().item():.2e}")
    print("\nFlash Attention working correctly!")

## Benchmark: Flash Attention vs Standard

In [ ]:
if TRITON_AVAILABLE:
    def benchmark_attention(seq_len, batch=2, heads=8, head_dim=64, n_iter=50):
        """Benchmark attention implementations."""
        q = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
        k = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
        v = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
        
        # Measure memory before
        torch.cuda.reset_peak_memory_stats()
        
        # Warmup
        for _ in range(5):
            _ = flash_attention(q, k, v)
        torch.cuda.synchronize()
        
        # Benchmark Flash
        torch.cuda.reset_peak_memory_stats()
        start = time.time()
        for _ in range(n_iter):
            _ = flash_attention(q, k, v)
        torch.cuda.synchronize()
        time_flash = (time.time() - start) / n_iter * 1000
        mem_flash = torch.cuda.max_memory_allocated() / 1e9
        
        # Benchmark standard
        scale = 1.0 / (head_dim ** 0.5)
        
        for _ in range(5):
            scores = torch.matmul(q, k.transpose(-2, -1)) * scale
            probs = torch.softmax(scores, dim=-1)
            _ = torch.matmul(probs, v)
        torch.cuda.synchronize()
        
        torch.cuda.reset_peak_memory_stats()
        start = time.time()
        for _ in range(n_iter):
            scores = torch.matmul(q, k.transpose(-2, -1)) * scale
            probs = torch.softmax(scores, dim=-1)
            _ = torch.matmul(probs, v)
        torch.cuda.synchronize()
        time_standard = (time.time() - start) / n_iter * 1000
        mem_standard = torch.cuda.max_memory_allocated() / 1e9
        
        return time_flash, time_standard, mem_flash, mem_standard
    
    # Benchmark different sequence lengths
    seq_lengths = [128, 256, 512, 1024, 2048]
    
    results = []
    print("Flash Attention vs Standard Attention\n")
    print(f"{'Seq Len':<10} {'Flash (ms)':<12} {'Std (ms)':<12} {'Speedup':<10} {'Flash Mem':<12} {'Std Mem':<12}")
    print("-" * 80)
    
    for seq_len in seq_lengths:
        time_flash, time_std, mem_flash, mem_std = benchmark_attention(seq_len)
        speedup = time_std / time_flash
        results.append((seq_len, time_flash, time_std, speedup, mem_flash, mem_std))
        
        print(f"{seq_len:<10} {time_flash:<12.2f} {time_std:<12.2f} {speedup:<10.2f}x {mem_flash:<12.2f} {mem_std:<12.2f}")

In [ ]:
if TRITON_AVAILABLE and results:
    # Visualize attention benchmarks
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    seq_lens = [r[0] for r in results]
    flash_times = [r[1] for r in results]
    std_times = [r[2] for r in results]
    speedups = [r[3] for r in results]
    flash_mems = [r[4] for r in results]
    std_mems = [r[5] for r in results]
    
    # Time comparison
    axes[0, 0].plot(seq_lens, flash_times, 'o-', label='Flash Attention', linewidth=2)
    axes[0, 0].plot(seq_lens, std_times, 's-', label='Standard Attention', linewidth=2)
    axes[0, 0].set_xlabel('Sequence Length')
    axes[0, 0].set_ylabel('Time (ms)')
    axes[0, 0].set_title('Attention Time Comparison')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # Speedup
    axes[0, 1].plot(seq_lens, speedups, 'o-', linewidth=2, markersize=8)
    axes[0, 1].axhline(y=1, color='r', linestyle='--', alpha=0.5)
    axes[0, 1].set_xlabel('Sequence Length')
    axes[0, 1].set_ylabel('Speedup (Standard / Flash)')
    axes[0, 1].set_title('Flash Attention Speedup')
    axes[0, 1].grid(alpha=0.3)
    
    # Memory comparison
    axes[1, 0].plot(seq_lens, flash_mems, 'o-', label='Flash Attention', linewidth=2)
    axes[1, 0].plot(seq_lens, std_mems, 's-', label='Standard Attention', linewidth=2)
    axes[1, 0].set_xlabel('Sequence Length')
    axes[1, 0].set_ylabel('Memory (GB)')
    axes[1, 0].set_title('Memory Usage Comparison')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    
    # Memory savings
    mem_savings = [1 - (f / s) for f, s in zip(flash_mems, std_mems)]
    axes[1, 1].bar(range(len(seq_lens)), mem_savings, alpha=0.8)
    axes[1, 1].set_xlabel('Sequence Length')
    axes[1, 1].set_ylabel('Memory Savings (%)')
    axes[1, 1].set_title('Flash Attention Memory Savings')
    axes[1, 1].set_xticks(range(len(seq_lens)))
    axes[1, 1].set_xticklabels(seq_lens)
    axes[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('flash_attention_benchmark.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nKey Insights:")
    print("1. Flash Attention speedup increases with sequence length")
    print("2. Memory savings dramatic for long sequences (O(N) vs O(N^2))")
    print("3. No quality loss - numerically equivalent to standard attention")
    print("4. Enables training with much longer sequences")

## When to Use Triton vs CUDA

### Use Triton When:

1. **Prototyping Custom Kernels**
   - Faster iteration (no separate compilation)
   - Python-like syntax
   - Good performance out of the box

2. **Kernel Fusion**
   - Combine multiple operations
   - Reduce memory traffic
   - Examples: LayerNorm + residual, attention patterns

3. **Research & Experimentation**
   - Trying new attention patterns
   - Custom normalization schemes
   - Novel operators

4. **Production (with caveats)**
   - PyTorch 2.0+ with torch.compile
   - Well-tested kernels
   - 90% of hand-tuned CUDA performance

### Use CUDA When:

1. **Maximum Performance Required**
   - Every microsecond counts
   - Fine-grained control needed
   - Production-critical paths

2. **Complex Synchronization**
   - Advanced shared memory patterns
   - Warp-level primitives
   - Hardware-specific optimizations

3. **Library Development**
   - cuBLAS, cuDNN equivalents
   - Broad hardware support
   - Mature, stable implementations

4. **Legacy Codebases**
   - Existing CUDA infrastructure
   - Team expertise in CUDA

### Performance Comparison

| Metric | Hand-tuned CUDA | Triton | PyTorch |
|--------|-----------------|--------|----------|
| Development time | Weeks | Days | Hours |
| Performance | 100% | 90-95% | 80-90% |
| Code complexity | High | Medium | Low |
| Maintainability | Low | Medium | High |
| Flexibility | Maximum | High | Medium |

## Summary

### Triton's Impact

1. **Democratizes GPU Programming**
   - ML researchers can write efficient kernels
   - No need to be CUDA expert
   - Fast experimentation cycle

2. **Powers Modern ML Systems**
   - Flash Attention (transformers)
   - PyTorch 2.0 compiler
   - Custom operations in research

3. **Bridges Research-Production Gap**
   - Prototype in Python
   - Deploy with good performance
   - Optimize critical paths later

### Key Takeaways

1. Triton = **Python-like GPU programming**
2. Automatic optimization of memory access patterns
3. 90-95% of CUDA performance with 10x less code
4. Ideal for kernel fusion and custom attention
5. Production-ready via PyTorch 2.0+

### Next Steps

- **Notebook 52**: Custom attention kernels (Flash Attention deep dive)
- **Notebook 53**: Fused operations for transformers
- **Notebook 54**: Quantization methods

### Resources

- [Triton Documentation](https://triton-lang.org/)
- [Flash Attention Paper](https://arxiv.org/abs/2205.14135)
- [Flash Attention 2 Paper](https://arxiv.org/abs/2307.08691)
- [Triton Tutorial](https://triton-lang.org/main/getting-started/tutorials/index.html)